In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 02 - Road Distance From OSM

The honest measurement, for free. We load the real drivable road network of central Copenhagen -- downloaded once from OpenStreetMap with OSMnx (keyless Overpass, no Docker, no server) and cached to GraphML -- snap points to the nearest road, and compute shortest-path **road distance** and **travel time** with NetworkX. Then we show what the crow misses: mode differences and one-way asymmetry.

In [2]:
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

from src.graph.build import load_city_graph
from src.graph.snap import SnapIndex
from src.distance.road import (
    add_edge_speeds, add_edge_travel_times, road_distance_m, road_travel_time_s,
)
from src.distance.haversine import haversine_m
from src.scenarios.points import load_scenario
from src.config import get_settings
from src.io_utils import save_fig, save_json

settings = get_settings()
scenario = load_scenario('copenhagen')
depot = tuple(scenario['depot'])
customers = [tuple(c) for c in scenario['customers']]
drive = load_city_graph('copenhagen', 'drive')
print('drive graph:', drive.number_of_nodes(), 'nodes,', drive.number_of_edges(), 'edges')

drive graph: 2321 nodes, 5349 edges


## The real road network

This is the graph the shortest path runs on.

In [3]:
fig, ax = ox.plot_graph(
    drive, node_size=0, edge_color='#5B8FF9', edge_linewidth=0.5,
    bgcolor='white', show=False, close=False,
)
ax.set_title('Central Copenhagen drivable road network (OpenStreetMap via OSMnx)')
save_fig(fig, '02_road_network')
plt.show()

## Snap points to the network, then route

Each point is snapped to the nearest road node; the snapping distance is an explicit caveat. Then NetworkX finds the shortest drivable path.

In [4]:
index = SnapIndex.from_graph(drive)
depot_node, depot_snap = index.nearest(*depot)
target = customers[0]
target_node, target_snap = index.nearest(*target)
route = nx.shortest_path(drive, depot_node, target_node, weight='length')
road_m = road_distance_m(drive, depot_node, target_node)
straight_m = haversine_m(depot[0], depot[1], target[0], target[1])
snap_info = {
    'depot_snap_m': depot_snap, 'target_snap_m': target_snap,
    'straight_m': straight_m, 'road_m': road_m,
    'detour_factor': road_m / straight_m,
}
save_json(snap_info, '02_route_example')
snap_info

{'depot_snap_m': 0.0,
 'target_snap_m': 0.0,
 'straight_m': 3024.6850302985595,
 'road_m': 3446.6500616518274,
 'detour_factor': 1.1395070981363031}

In [5]:
fig, ax = ox.plot_graph_route(
    drive, route, route_color='#E45756', route_linewidth=3, node_size=0,
    edge_color='#dddddd', edge_linewidth=0.5, bgcolor='white', show=False, close=False,
)
ax.set_title(f'Shortest drivable path: {road_m:.0f} m road vs {straight_m:.0f} m straight')
save_fig(fig, '02_shortest_route')
plt.show()

## Drive, walk, and bike give different distances

One-ways, pedestrian paths, and cycleways change the network, so the same two points are different distances apart by mode. We take a short hop across the Inner Harbour, from the city-centre side to Christianshavn, where a cars-no, bikes-and-feet-yes bridge makes the gap obvious. This needs the walk and bike graphs cached; run `make graph` on a live connection to populate them. The cell loads whatever modes are cached and compares them, so it stays offline-safe.

In [6]:
from src.graph.build import city_graph_path
mode_a = (55.6786, 12.5890)  # city-centre side, near Nyhavn
mode_b = (55.6736, 12.5940)  # Christianshavn, across the harbour
mode_straight = haversine_m(mode_a[0], mode_a[1], mode_b[0], mode_b[1])
modes = {}
for mode in ['drive', 'walk', 'bike']:
    path = city_graph_path('copenhagen', mode)
    if not path.exists():
        print('skip', mode, '- no cached graph (run `make graph`)')
        continue
    g = ox.load_graphml(path)
    idx = SnapIndex.from_graph(g)
    o, _ = idx.nearest(*mode_a)
    d, _ = idx.nearest(*mode_b)
    modes[mode] = road_distance_m(g, o, d)
save_json({'straight_m': mode_straight, **modes}, '02_mode_distances')
modes

skip walk - no cached graph (run `make graph`)
skip bike - no cached graph (run `make graph`)


{'drive': 1359.263409211645}

In [7]:
if len(modes) >= 2:
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = {'drive': '#4C78A8', 'walk': '#54A24B', 'bike': '#F58518'}
    ax.bar(list(modes), list(modes.values()), color=[colors[m] for m in modes])
    ax.axhline(mode_straight, color='#E45756', ls='--',
               label=f'straight line ({mode_straight:.0f} m)')
    ax.set_ylabel('distance (m)')
    ax.set_title('One hop across the harbour, by travel mode')
    ax.legend()
    fig.tight_layout()
    save_fig(fig, '02_mode_distances')
    plt.show()
else:
    print('Only', list(modes), 'cached; run `make graph` for walk/bike to draw this.')

Only ['drive'] cached; run `make graph` for walk/bike to draw this.


## Road distance is asymmetric; straight-line is not

On one-way streets A->B is not B->A. We scan depot<->customer pairs for the largest gap.

In [8]:
pairs = []
for lat, lon in customers[:40]:
    node, _ = index.nearest(lat, lon)
    try:
        there = road_distance_m(drive, depot_node, node)
        back = road_distance_m(drive, node, depot_node)
    except nx.NetworkXNoPath:
        continue
    pairs.append((abs(there - back), there, back, node))
pairs.sort(reverse=True)
worst = pairs[0]
asymmetry = {'max_gap_m': worst[0], 'there_m': worst[1], 'back_m': worst[2]}
save_json(asymmetry, '02_asymmetry')
asymmetry

{'max_gap_m': 733.0533199183128,
 'there_m': 2401.7839502095967,
 'back_m': 1668.730630291284}

## Free travel time from speed limits

Weight edges by speed limit, then shortest **time**.

In [9]:
for mode in ['drive']:
    add_edge_speeds(drive, settings.speeds.default_kph, settings.speeds.by_highway)
    add_edge_travel_times(drive)
time_s = road_travel_time_s(drive, depot_node, target_node)
timing = {'road_m': road_m, 'travel_time_s': time_s, 'travel_time_min': time_s / 60}
save_json(timing, '02_travel_time')
timing

{'road_m': 3446.6500616518274,
 'travel_time_s': 278.25926063717424,
 'travel_time_min': 4.637654343952904}

**Takeaway.** For nothing but a one-time download and some CPU, NetworkX on an OSM graph gives real road distance, mode-specific distance, directional asymmetry, and a free speed-limited travel time -- everything the straight line cannot see.